In [2]:
import os
import numpy as np
import pandas as pd
import pyodbc

# -----------------------------
# Config (NO pongas la clave aquí)
# -----------------------------
SERVER = "10.147.17.185"
PORT = 1433
USERNAME = "cmpcuser"

PASSWORD_ENV_VAR = "OTMS_SQL_PASSWORD_DEV"   # <- ÚNICO lugar donde se define el nombre
PASSWORD = os.getenv(PASSWORD_ENV_VAR)       # <- variable de entorno
DRIVER = "{ODBC Driver 17 for SQL Server}"

DB = "cmpc_20240925_093000"
SCHEMA = "dbo"
TABLE = "ypf_alarms"

TIME_COL_OVERRIDE = None   # ej: "EventDateTime"
LAST_N_DAYS = None           # None para todo histórico
GAP_MIN = 5
TS_CHUNKSIZE = 200_000

# -----------------------------
# Helpers
# -----------------------------
def connect():
    if not PASSWORD:
        raise RuntimeError(
            f"No encontré la variable de entorno {PASSWORD_ENV_VAR}. "
            f"En PowerShell: setx {PASSWORD_ENV_VAR} \"TU_PASSWORD\" y reinicia la terminal."
        )

    conn_str = (
        f"DRIVER={DRIVER};"
        f"SERVER={SERVER},{PORT};"
        f"DATABASE={DB};"
        f"UID={USERNAME};"
        f"PWD={PASSWORD};"
        "TrustServerCertificate=yes;"
    )

    print(f"Conectando a {SERVER}:{PORT} | DB={DB} | user={USERNAME} ...")
    return pyodbc.connect(conn_str)


def fetch_df(conn, query, params=None):
    return pd.read_sql(query, conn, params=params)


def get_columns_and_types(conn):
    q = f"""
    SELECT COLUMN_NAME, DATA_TYPE, IS_NULLABLE
    FROM [{DB}].INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = ? AND TABLE_NAME = ?
    ORDER BY ORDINAL_POSITION
    """
    return fetch_df(conn, q, params=[SCHEMA, TABLE])


def score_time_col(colname: str) -> int:
    c = colname.lower()
    score = 0
    if "event" in c: score += 5
    if "alarm" in c: score += 4
    if "time" in c or "timestamp" in c: score += 6
    if "date" in c or "datetime" in c: score += 5
    if "fecha" in c: score += 5
    if "hora" in c: score += 4
    if "ack" in c: score -= 2
    if "clear" in c: score -= 1
    if "update" in c or "updated" in c: score -= 1
    if "insert" in c or "created" in c: score -= 1
    return score


def pick_time_column(conn, df_cols: pd.DataFrame) -> str:
    columns = df_cols["COLUMN_NAME"].tolist()

    if TIME_COL_OVERRIDE:
        if TIME_COL_OVERRIDE not in columns:
            raise RuntimeError(
                f"TIME_COL_OVERRIDE='{TIME_COL_OVERRIDE}' no existe en la tabla. "
                f"Revisa el nombre exacto en 'Columnas encontradas'."
            )
        return TIME_COL_OVERRIDE

    time_types = {"datetime", "datetime2", "smalldatetime", "date", "time", "datetimeoffset"}
    candidates = df_cols[df_cols["DATA_TYPE"].str.lower().isin(time_types)].copy()

    if candidates.empty:
        name_candidates = [c for c in columns if any(k in c.lower() for k in ["time", "date", "fecha", "hora", "ts"])]
        if not name_candidates:
            raise RuntimeError(
                "No encontré columnas de tipo fecha/hora ni nombres que parezcan timestamp. "
                "Revisa las columnas impresas y fuerza TIME_COL_OVERRIDE."
            )
        return sorted(name_candidates, key=lambda x: score_time_col(x), reverse=True)[0]

    candidates["name_score"] = candidates["COLUMN_NAME"].apply(score_time_col)
    candidates = candidates.sort_values(["name_score"], ascending=False)

    top = candidates.head(5)["COLUMN_NAME"].tolist()
    full_table = f"[{DB}].[{SCHEMA}].[{TABLE}]"

    where_recent = ""
    params = []
    if LAST_N_DAYS is not None:
        where_recent = "WHERE {col} >= DATEADD(day, -?, GETDATE())"
        params = [LAST_N_DAYS]

    stats = []
    for col in top:
        q = f"""
        SELECT
            SUM(CASE WHEN [{col}] IS NOT NULL THEN 1 ELSE 0 END) AS non_nulls,
            MIN([{col}]) AS min_time,
            MAX([{col}]) AS max_time
        FROM {full_table}
        """ + (("\n" + where_recent.format(col=f"[{col}]")) if LAST_N_DAYS is not None else "")

        df = fetch_df(conn, q, params=params)
        non_nulls = int(df.loc[0, "non_nulls"]) if pd.notna(df.loc[0, "non_nulls"]) else 0
        min_t = df.loc[0, "min_time"]
        max_t = df.loc[0, "max_time"]
        stats.append((col, score_time_col(col), non_nulls, min_t, max_t))

    stats_sorted = sorted(stats, key=lambda x: (x[1], x[2]), reverse=True)
    chosen = stats_sorted[0][0]

    print("\n=== Candidatos de columna de tiempo (top 5) ===")
    print("col | name_score | non_nulls | min_time | max_time")
    for col, sc, nn, mn, mx in stats_sorted:
        print(f"{col} | {sc} | {nn} | {mn} | {mx}")

    return chosen


def build_where_recent(time_col: str):
    if LAST_N_DAYS is None:
        return "", []
    return f"WHERE [{time_col}] >= DATEADD(day, -?, GETDATE())", [LAST_N_DAYS]


def stream_blocks(conn, full_table: str, time_col: str):
    where_recent, params = build_where_recent(time_col)

    q = f"""
    SELECT [{time_col}] AS t
    FROM {full_table}
    {where_recent}
    ORDER BY [{time_col}] ASC
    """

    blocks = []
    last_t = None
    block_start = None
    block_end = None
    block_n = 0

    for chunk in pd.read_sql(q, conn, params=params, chunksize=TS_CHUNKSIZE):
        if chunk.empty:
            continue
        ts = pd.to_datetime(chunk["t"], errors="coerce").dropna()
        if ts.empty:
            continue

        for t in ts:
            if last_t is None:
                block_start = t
                block_end = t
                block_n = 1
                last_t = t
                continue

            gap_sec = (t - last_t).total_seconds()
            if gap_sec > GAP_MIN * 60:
                duration_min = (block_end - block_start).total_seconds() / 60.0
                blocks.append((block_start, block_end, block_n, duration_min))

                block_start = t
                block_end = t
                block_n = 1
            else:
                block_end = t
                block_n += 1

            last_t = t

    if last_t is not None and block_start is not None:
        duration_min = (block_end - block_start).total_seconds() / 60.0
        blocks.append((block_start, block_end, block_n, duration_min))

    return pd.DataFrame(blocks, columns=["start", "end", "n", "duration_min"])


def main():
    conn = connect()
    full_table = f"[{DB}].[{SCHEMA}].[{TABLE}]"

    df_cols = get_columns_and_types(conn)
    print("\n=== Columnas encontradas ===")
    print(df_cols.to_string(index=False))

    time_col = pick_time_column(conn, df_cols)
    print(f"\n>>> Columna de tiempo seleccionada: {time_col}")

    where_recent, params = build_where_recent(time_col)

    df_range = fetch_df(conn, f"""
        SELECT
            MIN([{time_col}]) AS min_time,
            MAX([{time_col}]) AS max_time,
            COUNT(1) AS total_rows
        FROM {full_table}
        {where_recent}
    """, params=params)

    print("\n=== Rango y volumen ===")
    print(df_range.to_string(index=False))

    df_minute = fetch_df(conn, f"""
        SELECT
            DATEADD(minute, DATEDIFF(minute, 0, [{time_col}]), 0) AS [minute],
            COUNT(1) AS alarms
        FROM {full_table}
        {where_recent}
        GROUP BY DATEADD(minute, DATEDIFF(minute, 0, [{time_col}]), 0)
        ORDER BY [minute]
    """, params=params)

    print("\n=== Tasa por minuto: percentiles ===")
    if len(df_minute) > 0:
        x = df_minute["alarms"].to_numpy()
        p = [50, 75, 90, 95, 97, 98, 99, 99.5, 99.9]
        pct = {f"p{str(pp).replace('.','_')}": float(np.percentile(x, pp)) for pp in p}
        print(pd.Series(pct).to_string())
        print(f"max_per_minute: {int(x.max())}")
    else:
        print("No hay datos para tasa por minuto.")

    print(f"\n=== Bloques preliminares (gap > {GAP_MIN} min) ===")
    blocks = stream_blocks(conn, full_table, time_col)

    if not blocks.empty:
        print(blocks[["n", "duration_min"]].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_string())
        print("\nTop 20 bloques por tamaño:")
        print(blocks.sort_values("n", ascending=False).head(20).to_string(index=False))
    else:
        print("No hay suficientes datos para formar bloques.")

    conn.close()


if __name__ == "__main__":
    main()

Conectando a 10.147.17.185:1433 | DB=cmpc_20240925_093000 | user=cmpcuser ...

=== Columnas encontradas ===
      COLUMN_NAME DATA_TYPE IS_NULLABLE
               id       int          NO
    ALARMDATETIME  datetime          NO
         COMPOUND  nvarchar         YES
            BLOCK  nvarchar         YES
              ANM  nvarchar         YES
        ALARMTYPE  nvarchar         YES
  TAG_DESCRIPTION  nvarchar         YES
ALARM_DESCRIPTION  nvarchar         YES
          ALM_RTN  nvarchar         YES
         PRIORITY       int         YES
       REAL_VALUE  nvarchar         YES
      ALARM_VALUE  nvarchar         YES
         ENG_UNIT  nvarchar         YES
             HIST  nvarchar         YES
         LOCATION  nvarchar         YES
              GRP  nvarchar         YES
               CP  nvarchar         YES
          TAGTYPE  nvarchar         YES
         ALARM_ID  nvarchar         YES
      CNI_NETWORK  nvarchar         YES
           TENTHS  nvarchar         YES
      fecha_

C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_36536\1757251373.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn, params=params)
C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_36536\1757251373.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn, params=params)



=== Candidatos de columna de tiempo (top 5) ===
col | name_score | non_nulls | min_time | max_time
ALARMDATETIME | 15 | 3935628 | 2024-01-10 00:00:00 | 2025-12-10 23:57:09
fecha_carga | 5 | 3935628 | 2026-02-13 17:34:14.887000 | 2026-02-14 00:27:30.280000

>>> Columna de tiempo seleccionada: ALARMDATETIME

=== Rango y volumen ===
  min_time            max_time  total_rows
2024-01-10 2025-12-10 23:57:09     3935628


C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_36536\1757251373.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn, params=params)



=== Tasa por minuto: percentiles ===
p50       12.0
p75       36.0
p90       72.0
p95      102.0
p97      126.0
p98      144.0
p99      180.0
p99_5    216.0
p99_9    345.0
max_per_minute: 7734

=== Bloques preliminares (gap > 5 min) ===


C:\Users\AndrésRios\AppData\Local\Temp\ipykernel_36536\1757251373.py:163: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  for chunk in pd.read_sql(q, conn, params=params, chunksize=TS_CHUNKSIZE):


                   n  duration_min
count    5566.000000   5566.000000
mean      707.083723     31.335016
std      5540.905775    128.184712
min         1.000000      0.000000
50%        21.000000      6.058333
75%        78.000000     19.629167
90%       429.000000     55.800000
95%      1638.250000    108.366667
99%     14269.800000    446.025000
max    147714.000000   3038.216667

Top 20 bloques por tamaño:
              start                 end      n  duration_min
2025-08-04 21:21:50 2025-08-07 00:00:03 147714   3038.216667
2025-07-02 13:15:56 2025-07-03 22:03:38 129426   1967.700000
2024-05-10 00:00:00 2024-05-11 01:44:22 127794   1544.366667
2025-05-05 20:24:28 2025-05-07 18:58:26 120312   2793.966667
2024-09-10 17:11:38 2024-09-12 00:48:15 118368   1896.616667
2024-10-10 00:00:00 2024-10-11 20:03:59  88662   2643.983333
2025-07-05 13:26:20 2025-07-07 00:01:41  82488   2075.350000
2025-04-07 23:54:15 2025-04-09 00:30:04  80385   1475.816667
2025-08-02 00:00:02 2025-08-02 19:00:4